In [126]:
import re
import unicodedata
from pathlib import Path
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random

In [127]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Bhavik\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [128]:
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words('english'))

In [129]:
def load_books(folder_path):
    all_text = []

    for filepath in sorted(Path(folder_path).glob("*.txt")):
        print(f"Loading: {filepath.name}")
        with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
            all_text.append(f.read())

    full_text = "\n".join(all_text)
    print(f"\nTotal characters loaded: {len(full_text):,}")
    return full_text


def normalize_unicode(text):
    text = unicodedata.normalize("NFC", text)
    text = text.encode("ascii", errors="ignore").decode("ascii")
    return text


def clean_text(text):
    text = normalize_unicode(text)
    text = text.lower()

    text = re.sub(r'chapter\s+\w+', '', text)
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\(.*?\)', '', text)

    text = re.sub(r'(\w+)-(\w+)', r'\1 \2', text)
    text = re.sub(r"[^a-z\s.!?']", ' ', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

- we have done hte splitting manually cause spacy was getting a memory error.

In [130]:
def split_into_sentences(text):
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 5]

    print(f"Total sentences: {len(sentences):,}")
    return sentences


def tokenize_sentences(sentences):
    tokenized = []

    for sentence in sentences:
        tokens = sentence.split()
        tokens = [t.replace("'", "") for t in tokens]
        tokens = [t for t in tokens if t not in STOPWORDS]
        tokens = [t for t in tokens if len(t) >= 2]

        if tokens:
            tokenized.append(tokens)

    total_words = sum(len(s) for s in tokenized)
    print(f"Total tokenized sentences: {len(tokenized):,}")
    print(f"Total word tokens: {total_words:,}")

    return tokenized

In [131]:
text = load_books(r"D:\stats\Deep_Learning\NLP\harry_text")

text = clean_text(text)

sentences = split_into_sentences(text)

tokenized_sentences = tokenize_sentences(sentences)

Loading: HPBook1.txt
Loading: HPBook2.txt
Loading: HPBook3.txt
Loading: HPBook4.txt
Loading: HPBook5.txt
Loading: HPBook6.txt
Loading: HPBook7.txt

Total characters loaded: 6,325,070
Total sentences: 84,254
Total tokenized sentences: 83,419
Total word tokens: 584,870


- lookup tables

In [132]:
vocab_size = 5000

words = [w for sent in tokenized_sentences for w in sent]
word_counts = Counter(words)

most_common = word_counts.most_common(vocab_size)

word2idx = {w: i for i, (w, _) in enumerate(most_common)}
idx2word = {i: w for w, i in word2idx.items()}


In [ ]:
def generate_pairs(tokenized_sentences, word2idx, window_size=2):
    pairs = []

    for sentence in tokenized_sentences:
        for i, target in enumerate(sentence):

            if target not in word2idx:
                continue

            target_id = word2idx[target]

            for j in range(-window_size, window_size + 1):
                if j == 0:
                    continue

                if 0 <= i + j < len(sentence):
                    context = sentence[i + j]

                    if context in word2idx:
                        context_id = word2idx[context]
                        pairs.append((target_id, context_id))

    return pairs

- Negative sampling table

In [134]:
def build_unigram_table(tokenized_sentences, word2idx):
    word_counts = Counter([w for sent in tokenized_sentences for w in sent])

    vocab_size = len(word2idx)
    freqs = np.zeros(vocab_size)

    for word, idx in word2idx.items():
        freqs[idx] = word_counts[word]

    freqs = freqs ** 0.75
    probs = freqs / np.sum(freqs)

    return probs

In [135]:
def prepare_batches(pairs, batch_size=512):
    random.shuffle(pairs)

    for i in range(0, len(pairs), batch_size):
        batch = pairs[i:i+batch_size]

        targets = torch.tensor([p[0] for p in batch])
        contexts = torch.tensor([p[1] for p in batch])

        yield targets, contexts

In [136]:
def get_negative_samples(probs, batch_size, k):
    neg_samples = np.random.choice(len(probs), size=(batch_size, k), p=probs)
    return torch.tensor(neg_samples)

In [137]:
class Word2VecNS(nn.Module):
    def __init__(self, vocab_size, embedding_dim=100):
        super().__init__()

        self.target_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.context_embeddings = nn.Embedding(vocab_size, embedding_dim)

    def forward(self, target, context, negative):
        target_emb = self.target_embeddings(target)
        context_emb = self.context_embeddings(context)
        neg_emb = self.context_embeddings(negative)

        pos_score = torch.sum(target_emb * context_emb, dim=1)
        neg_score = torch.bmm(neg_emb, target_emb.unsqueeze(2)).squeeze()

        return pos_score, neg_score

In [ ]:
import torch.nn.functional as F

def train_model_ns_bce(pairs, tokenized_sentences, word2idx,
                      embedding_dim=100, epochs=3, lr=0.001, k=5):

    vocab_size = len(word2idx)
    model = Word2VecNS(vocab_size, embedding_dim)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()

    probs = build_unigram_table(tokenized_sentences, word2idx)

    for epoch in range(epochs):
        total_loss = 0

        for targets, contexts in prepare_batches(pairs):
            batch_size = len(targets)
            negatives = get_negative_samples(probs, batch_size, k)

            pos_score, neg_score = model(targets, contexts, negatives)

            pos_labels = torch.ones_like(pos_score)             
            neg_labels = torch.zeros_like(neg_score)            

            loss_pos = loss_fn(pos_score, pos_labels)
            loss_neg = loss_fn(neg_score, neg_labels)

            loss = loss_pos + loss_neg

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

    return model

In [139]:
def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (
        np.linalg.norm(vec1) * np.linalg.norm(vec2)
    )


def find_similar(word, model, word2idx, idx2word, top_k=5):
    if word not in word2idx:
        return []

    embeddings = model.target_embeddings.weight.detach().numpy()
    word_vec = embeddings[word2idx[word]]

    sims = []

    for i in range(len(idx2word)):
        sim = cosine_similarity(word_vec, embeddings[i])
        sims.append((idx2word[i], sim))

    sims.sort(key=lambda x: x[1], reverse=True)

    return sims[1:top_k+1]

In [140]:
pairs = generate_pairs(tokenized_sentences, word2idx)


model = train_model_ns(pairs, len(word2idx))

print(find_similar("harry", model, word2idx, idx2word))
print(find_similar("voldemort", model, word2idx, idx2word))

Total training pairs: 1,521,960
Epoch 1, avg loss: 18.8683, total loss: 56095.39
Epoch 2, avg loss: 9.4708, total loss: 28156.80
Epoch 3, avg loss: 4.7870, total loss: 14231.77
Epoch 4, avg loss: 3.3627, total loss: 9997.41
Epoch 5, avg loss: 2.7221, total loss: 8092.70
Epoch 6, avg loss: 2.3495, total loss: 6984.94
Epoch 7, avg loss: 2.1085, total loss: 6268.71
Epoch 8, avg loss: 1.9499, total loss: 5797.18
Epoch 9, avg loss: 1.8372, total loss: 5461.88
Epoch 10, avg loss: 1.7584, total loss: 5227.68
Epoch 11, avg loss: 1.7008, total loss: 5056.60
Epoch 12, avg loss: 1.6565, total loss: 4924.89
Epoch 13, avg loss: 1.6192, total loss: 4813.83
Epoch 14, avg loss: 1.5915, total loss: 4731.55
Epoch 15, avg loss: 1.5678, total loss: 4661.08
Epoch 16, avg loss: 1.5459, total loss: 4595.96
Epoch 17, avg loss: 1.5284, total loss: 4543.88
Epoch 18, avg loss: 1.5120, total loss: 4495.21
Epoch 19, avg loss: 1.4981, total loss: 4453.97
Epoch 20, avg loss: 1.4863, total loss: 4418.84
Epoch 21, avg

In [141]:
test_words = [
    "harry", "ron", "hermione",
    "voldemort", "dumbledore",
    "hogwarts", "wand", "magic",
    "dark", "death","father"
]

for w in test_words:
    similar = find_similar(w, model, word2idx, idx2word)
    print(f"Top similar to '{w}': {[s[0] for s in similar]}")

Top similar to 'harry': ['ron', 'hermione', 'snape', 'neville', 'hagrid']
Top similar to 'ron': ['hermione', 'harry', 'fred', 'george', 'ginny']
Top similar to 'hermione': ['ron', 'ginny', 'harry', 'fred', 'george']
Top similar to 'voldemort': ['lord', 'voldemorts', 'dumbledore', 'death', 'destroy']
Top similar to 'dumbledore': ['snape', 'lupin', 'dumbledores', 'voldemort', 'fudge']
Top similar to 'hogwarts': ['school', 'students', 'year', 'house', 'permission']
Top similar to 'wand': ['elder', 'wands', 'hand', 'tapped', 'arm']
Top similar to 'magic': ['ministry', 'minister', 'statement', 'uses', 'learning']
Top similar to 'dark': ['arts', 'lords', 'defence', 'lord', 'messing']
Top similar to 'death': ['eaters', 'eater', 'voldemort', 'master', 'grief']
Top similar to 'father': ['parents', 'mother', 'james', 'regulus', 'sirius']


In [144]:
# Word Analogy: king - man + woman = queen
# Adapted for Harry Potter: word_a - word_b + word_c = ?

def solve_analogy(word_a, word_b, word_c, model, word2idx, idx2word, top_k=5):
    """
    Solve: word_a : word_b :: word_c : ?
    Using: embedding(word_c) - embedding(word_b) + embedding(word_a)
    """
    embeddings = model.target_embeddings.weight.detach().numpy()
    
    if word_a not in word2idx or word_b not in word2idx or word_c not in word2idx:
        return f"One or more words not in vocabulary"
    
    # Get embeddings
    vec_a = embeddings[word2idx[word_a]]
    vec_b = embeddings[word2idx[word_b]]
    vec_c = embeddings[word2idx[word_c]]
    
    # Analogy: word_a : word_b :: word_c : ?
    # Answer vector = vec_c - vec_b + vec_a
    answer_vec = vec_c - vec_b + vec_a
    answer_vec = answer_vec / (np.linalg.norm(answer_vec) + 1e-10)
    
    # Find nearest
    sims = []
    for i in range(len(idx2word)):
        sim = np.dot(answer_vec, embeddings[i] / (np.linalg.norm(embeddings[i]) + 1e-10))
        sims.append((idx2word[i], sim))
    
    sims.sort(key=lambda x: x[1], reverse=True)
    
    # Filter out input words
    result = [w for w, s in sims if w not in [word_a, word_b, word_c]][:top_k]
    
    return result


# Test analogies on Harry Potter
print("Word Analogies (like: king - man + woman = queen)\n")
print("=" * 60)

analogies = [
    ("dumbledore", "old", "young", "dumbledore : old :: young : ?"),
    ("harry", "boy", "girl", "harry : boy :: girl : ?"),
    ("voldemort", "evil", "good", "voldemort : evil :: good : ?"),
    ("wand", "magic", "muggle", "wand : magic :: muggle : ?"),
    ("hogwarts", "school", "home", "hogwarts : school :: home : ?"),
]

for a, b, c, description in analogies:
    result = solve_analogy(a, b, c, model, word2idx, idx2word, top_k=3)
    print(f"\n{description}")
    print(f"  Answer: {result}")


Word Analogies (like: king - man + woman = queen)


dumbledore : old :: young : ?
  Answer: ['lord', 'mundungus', 'enforcement']

harry : boy :: girl : ?
  Answer: ['ginny', 'hermione', 'ron']

voldemort : evil :: good : ?
  Answer: ['firm', 'hoping', 'calling']

wand : magic :: muggle : ?
  Answer: ['thestral', 'deliberately', 'cup']

hogwarts : school :: home : ?
  Answer: ['minister', 'headquarters', 'whinging']
